# AlphaEarth Embeddings Python Tutorial

## Clustering Example

This python was converted from the original javascript here:

https://developers.google.com/earth-engine/tutorials/community/satellite-embedding-01-introduction

In [ ]:
import ee
import geemap

# ee.Authenticate())
ee.Initialize()

Use the satellite basemap (Note: Map.setOptions is specific to Code Editor)

In Python, you'll typically use geemap or folium for visualization

In [ ]:
embeddings = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")

geometry = ee.Geometry.Polygon(
    [[[76.3978, 12.5521], [76.3978, 12.3550], [76.6519, 12.3550], [76.6519, 12.5521]]]
)

In [ ]:
year = 2024
start_date = ee.Date.fromYMD(year, 1, 1)
end_date = start_date.advance(1, "year")

filtered_embeddings = embeddings.filter(ee.Filter.date(start_date, end_date)).filter(
    ee.Filter.bounds(geometry)
)

In [ ]:
embeddings_image = filtered_embeddings.mosaic()
print("Satellite Embedding Image", embeddings_image.getInfo())

In [ ]:
n_samples = 1000
training = embeddings_image.sample(
    region=geometry, scale=10, numPixels=n_samples, seed=100
)
print(training.first().getInfo())

In [ ]:
# Function to train a model for desired number of clusters
def get_clusters(n_clusters):
    clusterer = ee.Clusterer.wekaKMeans(n_clusters).train(training)

    # Cluster the image
    clustered = embeddings_image.cluster(clusterer)
    return clustered


cluster3 = get_clusters(3)

cluster5 = get_clusters(5)

cluster10 = get_clusters(10)

To visualize the results in Python, you can use geemap:

In [ ]:
vis_params = {"min": -0.3, "max": 0.3, "bands": ["A01", "A16", "A09"]}

Map = geemap.Map()
Map.centerObject(geometry, 12)
Map.addLayer(embeddings_image.clip(geometry), vis_params, 'Embeddings Image')
Map.addLayer(cluster3.randomVisualizer().clip(geometry), {}, '3 clusters')
Map.addLayer(cluster5.randomVisualizer().clip(geometry), {}, '5 clusters')
Map.addLayer(cluster10.randomVisualizer().clip(geometry), {}, '10 clusters')
Map